> **中文阅读版**：本文件是 `15_lean_engine_parity.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 当前案例研究策略上的 LEAN 一致性

本 notebook 从当前真实策略审计中单独提取 LEAN 行。它使用由原生 LEAN 引擎生成并保留的结果，以及与之匹配的 ML4T Backtest 配置。共享输入在引擎执行前已冻结。

**学习目标**

- 识别哪些选定的资产类别是有效的 LEAN 比较对象
- 读取跨成交、估值和终值的 LEAN 一致性
- 解释所测策略上仅 LEAN 引擎的计时
- 将合成压力测试证据与真实策略等价性区分开

**本书参考**：第 16 章，第 16.3 节

## 设置

In [1]:
"""Current LEAN parity evidence."""

import json

import polars as pl
from IPython.display import Markdown, display

from utils.paths import get_chapter_dir

In [2]:
# Production defaults - Papermill injects overrides after this cell
ROUND_SECONDS = 3

In [3]:
AUDIT_PATH = get_chapter_dir(16) / "resources" / "framework_parity_audit.json"
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
lean = audit["frameworks"]["lean"]
LEAN_NAME = f"{lean['display_name']} {lean['version']}"
CASE_NAMES = {
    "etfs": "ETF allocation",
    "cme_futures": "CME futures",
    "crypto_perps_funding": "Crypto perpetual funding",
    "fx_pairs": "FX allocation (USD-quoted pairs)",
    "us_equities_panel": "US equity panel",
}

In [4]:
display(Markdown(f"**Pinned engine:** {LEAN_NAME} with ML4T profile `{lean['profile']}`"))

**Pinned engine:** LEAN 18001 with ML4T profile `lean`

## 1. 受支持的真实策略

ETF、加密货币永续合约（crypto-perpetual）、以美元报价的外汇以及美国股票面板（US equity-panel）工作负载需要 LEAN。CME 行对于这个特定的冻结数据包不受支持：它包含连续根代码价格（continuous root prices），但缺少构建原生 LEAN 期货订阅所需的带日期合约链和换月映射（roll map）。

In [5]:
lean_results = (
    pl.DataFrame(audit["real_strategy_records"])
    .filter(pl.col("framework") == "lean")
    .with_columns(pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"))
    .select(
        "strategy",
        "status",
        "fills",
        "valuations",
        "valuation_timestamps_match",
        "equity_gap",
        "equity_raw_gap",
        "terminal_gap",
        "terminal_raw_gap",
        "negative_control_detected",
    )
)

assert lean_results.height == 4
assert lean_results["status"].to_list() == ["pass"] * 4
assert lean_results["valuation_timestamps_match"].all()
assert lean_results["negative_control_detected"].all()

display(lean_results)

strategy,status,fills,valuations,valuation_timestamps_match,equity_gap,equity_raw_gap,terminal_gap,terminal_raw_gap,negative_control_detected
str,str,i64,i64,bool,str,str,str,str,bool
"""ETF allocation""","""pass""",2457,1995,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000""",true
"""Crypto perpetual funding""","""pass""",8408,2426,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000""",true
"""FX allocation (USD-quoted pair…","""pass""",279,2108,true,"""0.00""","""0.00000000""","""0.00""","""0.00000000""",true
"""US equity panel""","""pass""",54265,4027,true,"""0.00""","""0.00000460""","""0.00""","""0.00000420""",true


In [6]:
lean_unsupported = (
    pl.DataFrame(audit["unsupported_records"])
    .filter(pl.col("framework") == "lean")
    .with_columns(pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"))
    .select("strategy", "reason")
)
display(lean_unsupported)

strategy,reason
str,str
"""CME futures""","""the frozen workload has contin…"


该表报告每个受支持工作负载的完整成交和估值计数。LEAN 使用原生股票、加密期货和外汇证券。该审计不会仅仅为了增加一个 LEAN 行而将连续 CME 根代码转换为另一种工具。

## 2. 仅引擎计时

计时器在引擎调用前立即启动，并在其返回时停止。使用一次预热和十个进程隔离样本。输入加载、模型推理、目标构建、适配器准备、结果提取和报告均在计时区域之外。

In [7]:
lean_timing = (
    pl.DataFrame(audit["performance_records"])
    .filter(pl.col("framework") == "lean")
    .with_columns(
        pl.col("case_study").replace_strict(CASE_NAMES).alias("strategy"),
        pl.col("framework_median_seconds").round(ROUND_SECONDS).alias("lean_seconds"),
        pl.col("ml4t_median_seconds").round(ROUND_SECONDS).alias("ml4t_seconds"),
        pl.col("framework_to_ml4t_ratio").round(2).alias("lean_div_ml4t"),
    )
    .select("strategy", "lean_seconds", "ml4t_seconds", "lean_div_ml4t")
)
display(lean_timing)

strategy,lean_seconds,ml4t_seconds,lean_div_ml4t
str,f64,f64,f64
"""ETF allocation""",2.564,0.75,3.42
"""Crypto perpetual funding""",2.801,0.657,4.27
"""FX allocation (USD-quoted pair…",0.913,0.153,5.95
"""US equity panel""",47.683,26.368,1.81


该计时结果适用于这些固定版本、数据包和引擎边界。它不是一般性的 LEAN 性能声明。

## 3. 合成压力测试仍属诊断性质

In [8]:
lean_stress = (
    pl.DataFrame(audit["synthetic_stress"]["records"])
    .filter(pl.col("framework") == "lean")
    .select("intents", "fills", "trades", "terminal_value", "status")
)
display(lean_stress)

intents,fills,trades,terminal_value,status
i64,i64,i64,f64,str
427790,361297,191297,184538.13,"""pass"""


保留的压力测试行在生成输入上测试规模以及校准后的 LEAN 配置。上述受支持行提供了真实数据证据。